# Gradio GUI Demo on Colab

Notebook này:
- tải **YOLOv8** và **Faster R-CNN** từ GitHub LFS
- tạo GUI bằng **Gradio**
- chạy ổn định trên Colab với `share=True`

In [1]:
# 1) Install dependencies
!pip install -q gradio ultralytics opencv-python pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.3 MB/s eta 0:00:00


In [2]:
# 2) Download both models from GitHub LFS
GITHUB_USER = "thanh-industry"
GITHUB_REPO = "deep-learning-asm02"
BRANCH = "main"

YOLO_FILENAME = "model_yolov8.pt"
FRCNN_FILENAME = "model_fasterrcnn.pt"

YOLO_URL = f"https://media.githubusercontent.com/media/{GITHUB_USER}/{GITHUB_REPO}/{BRANCH}/04-demo/{YOLO_FILENAME}"
FRCNN_URL = f"https://media.githubusercontent.com/media/{GITHUB_USER}/{GITHUB_REPO}/{BRANCH}/04-demo/{FRCNN_FILENAME}"

!wget -O {YOLO_FILENAME} {YOLO_URL}
!wget -O {FRCNN_FILENAME} {FRCNN_URL}

--2026-04-09 02:01:26--  https://media.githubusercontent.com/media/thanh-industry/deep-learning-asm02/main/04-demo/model_yolov8.pt
Resolving media.githubusercontent.com (media.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to media.githubusercontent.com (media.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6232106 (5.9M) [application/octet-stream]
Saving to: ‘model_yolov8.pt’

model_yolov8.pt     100%[===================>]   5.94M  --.-KB/s    in 0.1s    

2026-04-09 02:01:27 (52.2 MB/s) - ‘model_yolov8.pt’ saved [6232106/6232106]

--2026-04-09 02:01:27--  https://media.githubusercontent.com/media/thanh-industry/deep-learning-asm02/main/04-demo/model_fasterrcnn.pt
Resolving media.githubusercontent.com (media.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to media.githubusercontent.com (media.githubusercontent.com)|185.199.108.133|:4

In [3]:
import torch
import torchvision
import numpy as np
import cv2
import gradio as gr

from PIL import Image
from torchvision.transforms import functional as F
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from ultralytics import YOLO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

CLASS_NAMES = ["background", "car", "bus", "motorbike", "bicycle"]
ID_TO_CLASS = {i: name for i, name in enumerate(CLASS_NAMES)}

COLORS = {
    "car": (0, 255, 0),
    "bus": (255, 0, 0),
    "motorbike": (0, 0, 255),
    "bicycle": (255, 255, 0),
}

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device: cpu


In [4]:
# 3) Load both models
yolo_model = YOLO("model_yolov8.pt")

def get_frcnn_model():
    weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn(weights=weights)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, len(CLASS_NAMES))
    model.load_state_dict(torch.load("model_fasterrcnn.pt", map_location=device))
    model.to(device)
    model.eval()
    return model

frcnn_model = get_frcnn_model()
print("Both models loaded successfully.")

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 146MB/s]


Both models loaded successfully.


In [5]:
def draw_yolo_predictions(image, threshold=0.6):
    results = yolo_model.predict(image, conf=threshold, verbose=False)
    result_img = results[0].plot()

    if isinstance(result_img, np.ndarray):
        return Image.fromarray(result_img[:, :, ::-1])
    return result_img


def draw_frcnn_predictions(image, threshold=0.6):
    img_rgb = np.array(image.copy())
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    image_tensor = F.to_tensor(image).to(device)

    with torch.no_grad():
        output = frcnn_model([image_tensor])[0]

    h, w = img_bgr.shape[:2]
    font_scale = max(1.0, h / 600)
    thickness = max(2, int(h / 250))
    box_thickness = max(3, int(h / 200))

    for box, score, label in zip(output["boxes"], output["scores"], output["labels"]):
        if float(score) < threshold:
            continue

        x1, y1, x2, y2 = map(int, box.tolist())
        class_name = ID_TO_CLASS[int(label)]
        text = f"{class_name} {float(score):.2f}"

        color_rgb = COLORS.get(class_name, (255, 255, 255))
        color_bgr = (color_rgb[2], color_rgb[1], color_rgb[0])

        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color_bgr, box_thickness)

        (tw, th), baseline = cv2.getTextSize(
            text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness
        )

        text_y = y1 - 10
        if text_y - th - baseline < 0:
            text_y = y1 + th + 10

        cv2.rectangle(
            img_bgr,
            (x1, text_y - th - baseline - 8),
            (x1 + tw + 10, text_y + 4),
            color_bgr,
            -1
        )

        cv2.putText(
            img_bgr,
            text,
            (x1 + 5, text_y - 2),
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            (0, 0, 0),
            thickness,
            cv2.LINE_AA
        )

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(img_rgb)

In [6]:
def run_demo(image, threshold, mode):
    if image is None:
        return None, None, None

    if mode == "YOLOv8":
        return image, draw_yolo_predictions(image, threshold), None

    if mode == "Faster R-CNN":
        return image, None, draw_frcnn_predictions(image, threshold)

    yolo_result = draw_yolo_predictions(image, threshold)
    frcnn_result = draw_frcnn_predictions(image, threshold)
    return image, yolo_result, frcnn_result

In [7]:
demo = gr.Interface(
    fn=run_demo,
    inputs=[
        gr.Image(type="pil", label="Upload Image"),
        gr.Slider(0.1, 0.9, value=0.6, step=0.05, label="Confidence Threshold"),
        gr.Radio(["YOLOv8", "Faster R-CNN", "Compare both"], value="Compare both", label="Mode"),
    ],
    outputs=[
        gr.Image(label="Original"),
        gr.Image(label="YOLOv8"),
        gr.Image(label="Faster R-CNN"),
    ],
    title="Vehicle Detection GUI",
    description="Compare YOLOv8 and Faster R-CNN on uploaded images.",
)

In [8]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4f9084411419004395.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
